# Qdrant RAG

In [19]:
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import ChatOpenAI
from IPython.display import display, Markdown, Latex


load_dotenv("../.env")


True

In [53]:
client = QdrantClient("http://localhost:6333")
COLLECTION_NAME = "llm-zoomcamp-rag"

class QdrantRAG:

    def __init__(self, llm, ss_embedding_model: str, template: str) -> None:
        self.model = llm
        self.embedding_model = ss_embedding_model
        self.client = QdrantClient("http://localhost:6333")
        self.prompt = template

    def search(self, query: str, limit: int  = 5):
        
        results = self.client.query_points(
        collection_name=COLLECTION_NAME,
        query=models.Document( #embed the query text locally with "jinaai/jina-embeddings-v2-small-en"
            text=query,
            model=self.embedding_model 
            ),
            limit=limit, # top closest matches
            with_payload=True #to get metadata in the results
        )
        
        return "\n ".join([i.payload["text"] for i in results.points])

    def query(self, question: str):
        search_results = self.search(question)
        chain = self.prompt | self.model
        response = chain.invoke(
            {
                "context": search_results, # elasticsearch 
                "question": question,
            }
        )
        # print(search_results)
        return question, search_results, response
        



In [54]:
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

rag = QdrantRAG(
    llm=llm, 
    ss_embedding_model="jinaai/jina-embeddings-v2-small-en",
    template=ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
                    Use only the facts from the CONTEXT when answering the QUESTION.
                    CONTEXT: 
                    {context}
                    """,
                ),
                ("human", "{question}"),
            ]
        )
)

question, semantic_search, response = rag.query(question="Explain to me how to use terraform")

# Question

In [55]:
Markdown(
    question
)

Explain to me how to use terraform

In [56]:
Markdown(
     semantic_search
)

You get this error because I run the command terraform init outside the working directory, and this is wrong.You need first to navigate to the working directory that contains terraform configuration files, and and then run the command.
 It is an internet connectivity error, terraform is somehow not able to access the online registry. Check your VPN/Firewall settings (or just clear cookies or restart your network). Try terraform init again after this, it should work.
 https://techcommunity.microsoft.com/t5/azure-developer-community-blog/configuring-terraform-on-windows-10-linux-sub-system/ba-p/393845
 If you are on the free trial account on GCP you will face this issue when trying to deploy the infrastructures with terraform. This service is not available for this kind of account.
The solution I found was to delete the load_balancer.tf file and to comment or delete the rows that differentiate it on the main.tf file. After this just do terraform destroy to delete any infrastructure created on the fail attempts and re-run the terraform apply.
Code on main.tf to comment/delete:
Line 166, 167, 168
 If you get the following error
You have to edit variables.tf on the gcp folder, set your project-id and region and zones properly. Then, run terraform apply again.
You can find correct regions/zones here: https://cloud.google.com/compute/docs/regions-zones
Deploying MAGE to GCP  with Terraform via the VM (2.2.7)
FYI - It can take up to 20 minutes to deploy the MAGE Terraform files if you are using a GCP Virtual Machine. It is normal, so don’t interrupt the process or think it’s taking too long. If you have, make sure you run a terraform destroy before trying again as you will have likely partially created resources which will cause errors next time you run `terraform apply`.
`terraform destroy` may not completely delete partial resources - go to Google Cloud Console and use the search bar at the top to search for the ‘app.name’ you declared in your variables.tf file; this will list all resources with that name - make sure you delete them all before running `terraform apply` again.
Why are my GCP free credits going so fast? MAGE .tf files - Terraform Destroy not destroying all Resources
I checked my GCP billing last night & the MAGE Terraform IaC didn't destroy a GCP Resource called Filestore as ‘mage-data-prep- it has been costing £5.01 of my free credits each day  I now have £151 left - Alexey has assured me that This amount WILL BE SUFFICIENT funds to finish the course. Note to anyone who had issues deploying the MAGE terraform code: check your billing account to see what you're being charged for (main menu - billing) (even if it's your free credits) and run a search for 'mage-data-prep' in the top bar just to be sure that your resources have been destroyed - if any come up delete them.

In [57]:
Markdown(
    response.content
)

To use Terraform, follow these general steps:

1. **Install Terraform**: First, download and install Terraform on your machine from the official Terraform website.

2. **Set Up Your Working Directory**: Create a directory for your Terraform project. This directory will contain all your Terraform configuration files.

3. **Write Configuration Files**: Create `.tf` files in your working directory. These files define the infrastructure you want to create. For example, you might have a `main.tf` file that specifies resources like virtual machines, networks, etc.

4. **Initialize Terraform**: Navigate to your working directory in the terminal and run `terraform init`. This command initializes your working directory, downloads necessary provider plugins, and sets up the backend for storing state.

5. **Plan Your Infrastructure**: Run `terraform plan` to see what changes Terraform will make to your infrastructure. This command shows a preview of the actions Terraform will take to reach the desired state defined in your configuration files.

6. **Apply Changes**: Execute `terraform apply` to create or update your infrastructure according to your configuration files. Terraform will prompt you to confirm the changes before applying them.

7. **Manage Infrastructure**: You can update your configuration files and run `terraform apply` again to make changes to your infrastructure. Use `terraform destroy` to remove all resources defined in your configuration files.

8. **Troubleshoot Issues**: If you encounter errors, check your configuration files for mistakes, ensure you are in the correct working directory, and verify your internet connectivity if Terraform cannot access online resources.

Remember, if you are using a GCP free trial account, some services may not be available, and you might need to adjust your configuration accordingly. Additionally, always check your billing account to ensure you are not incurring unexpected charges.